## Paths and settings


In [ ]:
from pathlib import Path
from datetime import datetime, timezone

# CHANGE THIS: the complete snapshot folder, containing CHECKSUMS.sha256 and stgnn/.
DATA_DIR = Path(r"D:/Github/SIPPolusi/docs/experiments/ds-v0.1.0")
# CHANGE THIS: a new folder for each run.
OUTPUT_DIR = Path.cwd() / "stgnn_runs" / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")

TARGET = "pm25"           # "pm25" or "no2"
PROTOCOL = "fixed_split"  # "fixed_split" (PM2.5 only) or "loso"
FOLD = None               # Set a fold number for LOSO; see splits/loso_folds.parquet.
DEVICE = "cpu"           # Or "cuda"; strict determinism may reject unsupported GPU operations.
SEED = 42
EPOCHS = 200
BATCH_SIZE = 4
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 20
GRADIENT_CLIP = 1.0
THREADS = 4
USE_GRAPH = True
USE_LOCAL = True

assert (DATA_DIR / "CHECKSUMS.sha256").is_file(), f"Change DATA_DIR: {DATA_DIR}"
assert not OUTPUT_DIR.exists(), f"Choose a new OUTPUT_DIR: {OUTPUT_DIR}"
print("Data:", DATA_DIR.resolve())
print("Output:", OUTPUT_DIR.resolve())

## Dependencies


In [ ]:
# Uncomment once if the notebook kernel is missing dependencies, then restart the kernel.
# %pip install "torch>=2.2,<3" "torch-geometric>=2.5,<3" "numpy>=1.26" "pandas>=2.1" "pyarrow>=15" "pyproj>=3.6" "matplotlib>=3.8"

import os
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")  # Before importing NumPy/PyTorch.
import hashlib
import json
import random
import time
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch_geometric
from torch import nn
from torch.nn import functional as F
from torch_geometric.nn import GATConv
from pyproj import Transformer

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(THREADS)
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
print({"torch": str(torch.__version__), "pyg": torch_geometric.__version__, "device": DEVICE})

## Features and preprocessing


In [ ]:
TEMPORAL_VALUES = ['no2_mol_m2','ntl','temperature_2m','relative_humidity_2m',
    'wind_speed_10m','wind_direction_10m','boundary_layer_height','precipitation',
    'surface_pressure','geoscf_pm25_ugm3','geoscf_no2_ugm3']
TEMPORAL_AGES = ['no2_mol_m2_age_hours','ntl_age_hours','geoscf_pm25_age_hours','geoscf_no2_age_hours',
    'weather_age_hours']
LAND_VALUES = ['ndvi','ndbi']
LAND_AGES = ['ndvi_age_hours','ndbi_age_hours']
SENSOR_FEATURES = ['dist_nearest_sensor_m']

"""Train-fitted transformations with explicit masks; raw datasets are never imputed."""

import numpy as np


TEMPORAL_COLUMNS = [*TEMPORAL_VALUES, *TEMPORAL_AGES]
LOCAL_COLUMNS = [*LAND_VALUES, *LAND_AGES, *SENSOR_FEATURES]
TRANSFORMED_TEMPORAL = [name for c in TEMPORAL_COLUMNS for name in
                        (["wind_direction_sin", "wind_direction_cos"] if c == "wind_direction_10m" else [c])]


def transform_values(values, columns):
    values = np.asarray(values, dtype=np.float64)
    if values.shape[-1] != len(columns):
        raise ValueError("Input width differs from feature contract")
    out = []
    for i, name in enumerate(columns):
        x = np.where(np.isfinite(values[..., i]), values[..., i], np.nan)
        if name.endswith("_age_hours") or name == "dist_nearest_sensor_m":
            if np.any(x < 0):
                raise ValueError(f"Negative age/distance: {name}")
            x = np.log1p(x)
        if name == "wind_direction_10m":
            theta = np.deg2rad(x)
            out.extend([np.sin(theta), np.cos(theta)])
        else:
            out.append(x)
    return np.stack(out, axis=-1)


class FeatureScaler:
    def __init__(self, columns):
        self.columns = list(columns)
        self.mean = self.scale = self.count = None

    def fit(self, values):
        transformed = transform_values(values, self.columns)
        x = transformed.reshape(-1, transformed.shape[-1])
        mask = np.isfinite(x)
        self.count = mask.sum(axis=0)
        safe_count = np.maximum(self.count, 1)
        self.mean = np.where(mask, x, 0).sum(axis=0) / safe_count
        variance = np.where(mask, (x - self.mean) ** 2, 0).sum(axis=0) / safe_count
        self.scale = np.sqrt(variance)
        self.scale[self.scale < 1e-8] = 1.0
        return self

    def transform(self, values, columns):
        if list(columns) != self.columns or self.mean is None:
            raise ValueError("Unfitted scaler or feature order mismatch")
        x = transform_values(values, self.columns)
        mask = np.isfinite(x)
        scaled = np.where(mask, (x - self.mean) / self.scale, 0)
        result = np.concatenate([scaled, mask.astype(np.float64)], axis=-1).astype("float32")
        if not np.isfinite(result).all():
            raise ValueError("Nonfinite transformed f")
        return result

    def state(self):
        return dict(columns=self.columns, mean=self.mean.tolist(), scale=self.scale.tolist(), count=self.count.tolist())

    @classmethod
    def from_state(cls, state):
        obj = cls(state["columns"])
        obj.mean = np.asarray(state["mean"], dtype=float)
        obj.scale = np.asarray(state["scale"], dtype=float)
        obj.count = np.asarray(state["count"], dtype=int)
        if np.any(obj.scale <= 0) or not np.isfinite(obj.mean).all() or not np.isfinite(obj.scale).all():
            raise ValueError("Invalid preprocessing statistics")
        return obj


class Preprocessor:
    def fit(self, temporal, local, targets):
        self.temporal = FeatureScaler(TEMPORAL_COLUMNS).fit(temporal)
        self.local = FeatureScaler(LOCAL_COLUMNS).fit(local)
        y = np.asarray(targets, dtype=float)
        if not y.size or not np.isfinite(y).all():
            raise ValueError("Fit target statistics on finite training labels only")
        self.target_mean = float(y.mean())
        self.target_scale = max(float(y.std()), 1e-8)
        return self

    def transform(self, temporal, local, temporal_columns=TEMPORAL_COLUMNS, local_columns=LOCAL_COLUMNS):
        return (self.temporal.transform(temporal, temporal_columns),
                self.local.transform(local, local_columns))

    def target(self, y):
        return (y - self.target_mean) / self.target_scale

    def inverse_target(self, z, clip=True):
        raw = np.asarray(z) * self.target_scale + self.target_mean
        return np.maximum(raw, 0) if clip else raw

    def state(self):
        return dict(temporal=self.temporal.state(), local=self.local.state(),
                    target_mean=self.target_mean, target_scale=self.target_scale,
                    transformed_temporal_columns=TRANSFORMED_TEMPORAL,
                    output_policy="inverse target scaling then clamp at zero")

    @classmethod
    def from_state(cls, state):
        if state["transformed_temporal_columns"] != TRANSFORMED_TEMPORAL:
            raise ValueError("Unsupported transformed feature contract")
        obj = cls()
        obj.temporal = FeatureScaler.from_state(state["temporal"])
        obj.local = FeatureScaler.from_state(state["local"])
        obj.target_mean = float(state["target_mean"])
        obj.target_scale = float(state["target_scale"])
        if not np.isfinite(obj.target_mean) or not np.isfinite(obj.target_scale) or obj.target_scale <= 0:
            raise ValueError("Invalid target statistics")
        return obj

## Model


In [ ]:
"""Shared temporal/local encoders, graph attention, and scalar node regression."""

from dataclasses import dataclass

import torch
from torch import nn
from torch.nn import functional as F
from torch_geometric.nn import GATConv


@dataclass(frozen=True)
class ModelConfig:
    temporal_channels: int = 34
    local_channels: int = 10
    hidden_channels: int = 32
    local_hidden_channels: int = 16
    heads: int = 4
    kernel_size: int = 3
    dilations: tuple[int, ...] = (1, 2, 4)
    dropout: float = 0.1
    use_graph: bool = True
    use_local: bool = True

    def __post_init__(self):
        if min(self.temporal_channels, self.local_channels, self.hidden_channels,
               self.local_hidden_channels, self.heads, self.kernel_size) < 1:
            raise ValueError("Model dimensions must be positive")
        if self.hidden_channels % self.heads or not 0 <= self.dropout < 1:
            raise ValueError("Hidden width must divide into heads; dropout must be in [0, 1)")
        if not self.dilations or any(d < 1 for d in self.dilations):
            raise ValueError("Require positive temporal dilations")


class CausalConv1d(nn.Conv1d):
    def forward(self, x):
        left = self.dilation[0] * (self.kernel_size[0] - 1)
        return super().forward(F.pad(x, (left, 0)))


class TemporalBlock(nn.Module):
    def __init__(self, width, kernel_size, dilation, dropout):
        super().__init__()
        self.layers = nn.Sequential(
            CausalConv1d(width, width, kernel_size, dilation=dilation), nn.ReLU(), nn.Dropout(dropout),
            CausalConv1d(width, width, kernel_size, dilation=dilation), nn.ReLU(), nn.Dropout(dropout))

    def forward(self, x):
        return F.relu(x + self.layers(x))


class TemporalEncoder(nn.Module):
    """Input/output [batch, nodes, hours, channels]; convolution touches time only."""

    def __init__(self, config):
        super().__init__()
        self.projection = nn.Linear(config.temporal_channels, config.hidden_channels)
        self.blocks = nn.Sequential(*[
            TemporalBlock(config.hidden_channels, config.kernel_size, d, config.dropout)
            for d in config.dilations])

    def forward(self, x):
        b, n, hours, _ = x.shape
        h = self.projection(x).reshape(b * n, hours, -1).transpose(1, 2)
        return self.blocks(h).transpose(1, 2).reshape(b, n, hours, -1)


def batch_edges(edge_index, nodes, batches):
    """Disjoint copies of one graph, with one self loop per node in each copy."""
    if edge_index.dtype != torch.long or edge_index.ndim != 2 or edge_index.shape[0] != 2:
        raise ValueError("edge_index must be int64 [2, E]")
    if edge_index.numel() and (edge_index.min() < 0 or edge_index.max() >= nodes):
        raise ValueError("Edge endpoint outside graph")
    edges = edge_index[:, edge_index[0] != edge_index[1]]
    loops = torch.arange(nodes, device=edges.device).repeat(2, 1)
    edges = torch.unique(torch.cat([edges, loops], dim=1), dim=1)
    offsets = torch.arange(batches, device=edges.device) * nodes
    return (edges[:, None, :] + offsets[None, :, None]).reshape(2, -1)


class STGNN(nn.Module):
    """Returns standardized concentration [B, N, 1]; no data access or target scaling."""

    def __init__(self, config=ModelConfig()):
        super().__init__()
        self.config = config
        self.temporal = TemporalEncoder(config)
        h, s = config.hidden_channels, config.local_hidden_channels
        self.local = nn.Sequential(nn.Linear(config.local_channels, h), nn.ReLU(),
                                   nn.Dropout(config.dropout), nn.Linear(h, s), nn.ReLU())
        self.fusion = nn.Sequential(nn.Linear(h + s, h), nn.ReLU())
        self.graph = (GATConv(h, h // config.heads, heads=config.heads, concat=True,
                              dropout=config.dropout, add_self_loops=False)
                      if config.use_graph else None)
        self.norm = nn.LayerNorm(h)
        self.regression = nn.Sequential(nn.Linear(h, 16), nn.ReLU(), nn.Linear(16, 1))

    def forward(self, temporal, local, edge_index):
        c = self.config
        if temporal.ndim != 4 or temporal.shape[-1] != c.temporal_channels:
            raise ValueError("Invalid temporal shape")
        b, n, hours, _ = temporal.shape
        if min(b, n, hours) < 1 or local.shape != (b, n, c.local_channels):
            raise ValueError("Invalid local shape or empty batch")
        if not torch.isfinite(temporal).all() or not torch.isfinite(local).all():
            raise ValueError("Preprocess missing/nonfinite values before model inference")
        dynamic = self.temporal(temporal)[:, :, -1]
        static = self.local(local) if c.use_local else dynamic.new_zeros(b, n, c.local_hidden_channels)
        fused = self.fusion(torch.cat([dynamic, static], dim=-1))
        if self.graph is not None:
            edges = batch_edges(edge_index, n, b)
            spatial = self.graph(fused.reshape(b * n, -1), edges).reshape(b, n, -1)
            fused = fused + F.elu(spatial)
        return self.regression(self.norm(fused))

In [ ]:
model_config = ModelConfig(use_graph=USE_GRAPH, use_local=USE_LOCAL)
model = STGNN(model_config).to(DEVICE)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Temporal features:", TEMPORAL_COLUMNS)
print("Local features:", LOCAL_COLUMNS)

## Verify snapshot


In [ ]:
def stgnn_hash(path):
    result=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):
            result.update(block)
    return result.hexdigest()

def verify_snapshot(root):
    """Verify the frozen checksum list before reading training inputs."""
    root = Path(root).resolve()
    hashes = {}
    for line in (root / "CHECKSUMS.sha256").read_text(encoding="utf-8").splitlines():
        digest, name = line.split(maxsplit=1)
        path = (root / name).resolve()
        if not path.is_relative_to(root):
            raise ValueError(f"Snapshot path outside root: {name}")
        actual = stgnn_hash(path)
        # Git may check text out as CRLF on Windows. Accept only this text-only
        # conversion; Parquet and other binary inputs must match byte for byte.
        if actual != digest and path.suffix in (".md", ".json", ".toml"):
            actual = hashlib.sha256(path.read_bytes().replace(b"\r\n", b"\n")).hexdigest()
        if actual != digest:
            raise ValueError(f"Snapshot checksum mismatch: {name}")
        hashes[name] = digest
    required = {"labels.parquet", "stations.parquet", "split_spec.json", "stgnn/features.parquet",
                "stgnn/nodes.parquet", "stgnn/edges.parquet", "stgnn/manifest.json",
                "splits/train.parquet", "splits/val.parquet", "splits/test.parquet",
                "splits/loso_folds.parquet"}
    if not required <= hashes.keys():
        raise ValueError("Incomplete frozen snapshot checksum list")
    return hashes

hashes = verify_snapshot(DATA_DIR)
print("Verified files:", len(hashes))

## Check settings


In [ ]:
if TARGET not in ("pm25", "no2"):
    raise ValueError("Choose pm25 or no2")
if PROTOCOL not in ("fixed_split", "loso"):
    raise ValueError("Choose fixed_split or loso")
if PROTOCOL == "fixed_split" and (TARGET != "pm25" or FOLD is not None):
    raise ValueError("Fixed split requires PM2.5 and FOLD = None")
if PROTOCOL == "loso" and FOLD is None:
    raise ValueError("Set FOLD for LOSO")

## Load graph and stations


In [ ]:
manifest = json.loads((DATA_DIR / "stgnn/manifest.json").read_text(encoding="utf-8"))
if manifest["status"] != "complete" or manifest["horizon_hours"] != 0:
    raise ValueError("Require a completed estimation dataset with horizon zero")
window = int(manifest["window"])
if window != 24:
    raise ValueError("Architecture v1 requires a 24-hour history")
nodes = pd.read_parquet(DATA_DIR / "stgnn/nodes.parquet").sort_values("node_index")
if not np.array_equal(nodes.node_index, np.arange(len(nodes))) or not nodes.grid_id.is_unique:
    raise ValueError("Require contiguous canonical node indices and unique grid IDs")
edges = pd.read_parquet(DATA_DIR / "stgnn/edges.parquet")
edge_index = torch.tensor(edges[["source_node", "target_node"]].to_numpy().T, dtype=torch.long)
stations = pd.read_parquet(DATA_DIR / "stations.parquet")
labels = pd.read_parquet(DATA_DIR / "labels.parquet")

## Select train, validation, and test labels


In [ ]:
target_column = f"{TARGET}_ugm3"
held_out_group = None
if PROTOCOL == "fixed_split":
    training_stations = stations[stations.station_split == "train"]
    split_labels = {}
    for split in ("train", "val", "test"):
        table = pd.read_parquet(DATA_DIR / "splits" / f"{split}.parquet")
        split_labels[split] = table[(table.target == TARGET) & (table.protocol == "fixed_split")].copy()
else:
    folds = pd.read_parquet(DATA_DIR / "splits/loso_folds.parquet")
    held = folds[(folds.target == TARGET) & (folds.fold == FOLD)]
    if held.empty or held.group_id.nunique() != 1:
        raise ValueError("Unknown LOSO fold")
    held_out_group = str(held.group_id.iloc[0])
    training_stations = stations[stations.group_id != held_out_group]
    split_labels = {}
    for split in ("train", "val", "test"):
        eligible = (labels.group_id == held_out_group if split == "test"
                    else labels.group_id != held_out_group)
        table = labels[eligible & (labels.period == split) & labels[target_column].notna()].copy()
        split_labels[split] = table.rename(columns={target_column: "value_ugm3"})

## Check labels


In [ ]:
for split, table in split_labels.items():
    if table.empty or table.duplicated(["station_uuid", "time_utc"]).any():
        raise ValueError(f"Empty or duplicated {split} labels for this target/protocol")
    if not np.isfinite(table.value_ugm3).all() or not table.node_index.between(0, len(nodes) - 1).all():
        raise ValueError("Invalid target value or label node index")
    table["time_utc"] = pd.to_datetime(table.time_utc, utc=True)
    split_labels[split] = table.sort_values(["time_utc", "station_uuid"]).reset_index(drop=True)

## Load feature table


In [ ]:
feature_columns = ["time_utc", "node_index", "grid_id", *TEMPORAL_COLUMNS, *LOCAL_COLUMNS]
features = pd.read_parquet(DATA_DIR / "stgnn/features.parquet", columns=feature_columns)
if features.duplicated(["node_index", "time_utc"]).any():
    raise ValueError("Duplicate node/hour features")

## Arrange hourly features


In [ ]:
hours = pd.date_range(manifest["start_utc"], manifest["end_utc_exclusive"],
                           freq="h", inclusive="left")
index = pd.MultiIndex.from_product([hours, nodes.node_index], names=["time_utc", "node_index"])
if len(features) != len(index):
    raise ValueError("Incomplete feature history")
features = features.set_index(["time_utc", "node_index"]).reindex(index)
expected_grid_ids = np.tile(nodes.grid_id.to_numpy(), len(hours))
if features.grid_id.isna().any() or not np.array_equal(features.grid_id.to_numpy(), expected_grid_ids):
    raise ValueError("Feature grid/node mapping differs from graph")
shape = (len(hours), len(nodes), -1)
temporal_features = features[TEMPORAL_COLUMNS].to_numpy(dtype="float32").reshape(shape)
local_features = features[LOCAL_COLUMNS].to_numpy(dtype="float32").reshape(shape).copy()

## Distance to training stations


In [ ]:
# Use only training stations for sensor distance.
sensors = training_stations.rename(columns={"lng": "station_longitude", "lat": "station_latitude"})
sensors = sensors.drop_duplicates(["station_longitude", "station_latitude"])
sensor_x, sensor_y = Transformer.from_crs("EPSG:4326", "EPSG:32748", always_xy=True).transform(
    sensors.station_longitude.to_numpy(), sensors.station_latitude.to_numpy())
if len(sensors):
    dx = nodes.easting.to_numpy()[:, None] - np.asarray(sensor_x)[None, :]
    dy = nodes.northing.to_numpy()[:, None] - np.asarray(sensor_y)[None, :]
    local_features[:, :, -1] = np.sqrt(dx * dx + dy * dy).min(axis=1).astype("float32")
else:
    local_features[:, :, -1] = np.nan

## Select nodes for training statistics


In [ ]:
hour_index = {t: i for i, t in enumerate(hours)}
grid_coordinates = nodes.grid_id.str.extract(r"^r(\d+)_c(\d+)$").astype(int).to_numpy()
training_mask = np.zeros(len(nodes), dtype=bool)
for grid_id in training_stations.grid_id.unique():
    row, column = map(int, grid_id[1:].split("_c"))
    training_mask |= (np.abs(grid_coordinates - [row, column]).max(axis=1) <= 1)
excluded = stations[~stations.uuid.isin(training_stations.uuid)]
for grid_id in excluded.grid_id.unique():
    row, column = map(int, grid_id[1:].split("_c"))
    training_mask &= ~(np.abs(grid_coordinates - [row, column]).max(axis=1) <= 1)
training_nodes = np.flatnonzero(training_mask)
if not training_mask.any():
    raise ValueError("No eligible feature-fit nodes")

## Check history and show splits


In [ ]:
for table in split_labels.values():
    positions = [hour_index.get(t, -1) for t in table.time_utc]
    if min(positions) < window - 1:
        raise ValueError("Label lacks a complete causal input window")

split_hours = {split: table.time_utc.drop_duplicates().tolist() for split, table in split_labels.items()}
display(pd.DataFrame({split: {"labels": len(table), "hours": len(split_hours[split]),
    "stations": table.station_uuid.nunique()} for split, table in split_labels.items()}).T)
print("Graph:", len(nodes), "nodes,", edge_index.shape[1], "edges")

## Fit training statistics


In [ ]:
train_labels = split_labels["train"]
first = hour_index[train_labels.time_utc.min()] - window + 1
last = hour_index[train_labels.time_utc.max()] + 1
fit_times = [str(hours[first]), str(hours[last - 1])]
preprocessing = Preprocessor().fit(
    temporal_features[first:last, training_nodes],
    local_features[first:last, training_nodes],
    train_labels.value_ugm3.to_numpy())
print("Feature fit hours:", fit_times)
print("Target mean/std:", preprocessing.target_mean, preprocessing.target_scale)

## Batch preparation


In [ ]:
def make_batch(hours, split):
        temporal, local, labels = [], [], []
        for b, hour in enumerate(hours):
            hour = pd.Timestamp(hour)
            i = hour_index[hour]
            temporal.append(temporal_features[i - window + 1:i + 1].transpose(1, 0, 2))
            local.append(local_features[i])
            table = split_labels[split]
            labels.append(table[table.time_utc == hour].assign(batch_index=b))
        raw_t, raw_s = np.stack(temporal), np.stack(local)
        t, s = preprocessing.transform(raw_t, raw_s)
        lab = pd.concat(labels, ignore_index=True)
        return dict(temporal=torch.from_numpy(t).to(DEVICE), local=torch.from_numpy(s).to(DEVICE),
                    edge_index=edge_index.to(DEVICE),
                    label_batch=torch.tensor(lab.batch_index.to_numpy(), dtype=torch.long, device=DEVICE),
                    label_node=torch.tensor(lab.node_index.to_numpy(dtype="int64"), dtype=torch.long, device=DEVICE),
                    label_values=torch.tensor(lab.value_ugm3.to_numpy(), dtype=torch.float32, device=DEVICE),
                    labels=lab)

## Validation


In [ ]:
@torch.no_grad()
def validation_rmse():
    model.eval()
    squared_error, count = 0.0, 0
    for start in range(0, len(split_hours["val"]), BATCH_SIZE):
        batch = make_batch(split_hours["val"][start:start + BATCH_SIZE], "val")
        output = model(batch["temporal"], batch["local"], batch["edge_index"])
        z = output[batch["label_batch"], batch["label_node"], 0].cpu().numpy()
        predicted = preprocessing.inverse_target(z)
        observed = batch["labels"].value_ugm3.to_numpy()
        if not np.isfinite(predicted).all():
            raise ValueError("Nonfinite validation predictions")
        squared_error += float(np.sum((predicted - observed) ** 2))
        count += len(observed)
    return (squared_error / count) ** 0.5

## Training setup


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
rng = np.random.default_rng(SEED)
history = []
best_rmse, best_epoch, stale = float("inf"), 0, 0
best_weights = None

## Train


In [ ]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_hours = rng.permutation(split_hours["train"]).tolist()
    loss_sum, count = 0.0, 0

    for start in range(0, len(train_hours), BATCH_SIZE):
        batch = make_batch(train_hours[start:start + BATCH_SIZE], "train")
        optimizer.zero_grad(set_to_none=True)
        output = model(batch["temporal"], batch["local"], batch["edge_index"])
        prediction = output[batch["label_batch"], batch["label_node"], 0]
        target = preprocessing.target(batch["label_values"])
        loss = ((prediction - target) ** 2).mean()
        if not torch.isfinite(loss):
            raise ValueError("Nonfinite training loss")
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP, error_if_nonfinite=True)
        optimizer.step()
        loss_sum += loss.item() * len(target)
        count += len(target)

    rmse = validation_rmse()
    history.append(dict(epoch=epoch, train_mse_standardized=loss_sum / count, validation_rmse=rmse))
    print(f"Epoch {epoch:03d} | train MSE {loss_sum / count:.4f} | val RMSE {rmse:.4f}", flush=True)

    if rmse < best_rmse:
        best_rmse, best_epoch, stale = rmse, epoch, 0
        best_weights = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        stale += 1
    if stale >= PATIENCE:
        print("Early stopping")
        break

## Save results


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
metadata = dict(target=TARGET, protocol=PROTOCOL, fold=FOLD,
    snapshot_hashes=hashes, unit="ug/m3", seed=SEED,
    best_epoch=best_epoch, validation_rmse=best_rmse)
torch.save(dict(artifact_version=1, model_config=asdict(model_config),
    state_dict=best_weights, preprocessing=preprocessing.state(), metadata=metadata),
    OUTPUT_DIR / "best.pt")
pd.DataFrame(history).to_csv(OUTPUT_DIR / "history.csv", index=False)
(OUTPUT_DIR / "preprocessing.json").write_text(json.dumps(preprocessing.state(), indent=2), encoding="utf-8")
print("Best epoch:", best_epoch)
print("Saved:", OUTPUT_DIR.resolve())

## Learning curves


In [ ]:
history_frame = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history_frame.epoch, history_frame.train_mse_standardized)
axes[0].set(xlabel="Epoch", ylabel="Training MSE (standardized)")
axes[1].plot(history_frame.epoch, history_frame.validation_rmse)
axes[1].set(xlabel="Epoch", ylabel="Validation RMSE (ug/m3)")
plt.tight_layout()
plt.show()

## Load best checkpoint


In [ ]:
artifact = torch.load(OUTPUT_DIR / "best.pt", map_location=DEVICE, weights_only=True)
model.load_state_dict(artifact["state_dict"])
preprocessing = Preprocessor.from_state(artifact["preprocessing"])
model.eval()

## Predict validation and test


In [ ]:
prediction_tables = {}

# Evaluate test only after validation has selected the best model.
for split in ("val", "test"):
    split_frames = []
    hours_to_predict = split_hours[split]

    with torch.no_grad():
        for start in range(0, len(hours_to_predict), BATCH_SIZE):
            batch_hours = hours_to_predict[start:start + BATCH_SIZE]
            batch = make_batch(batch_hours, split)
            node_predictions = model(batch["temporal"], batch["local"], batch["edge_index"])
            station_predictions = node_predictions[batch["label_batch"], batch["label_node"], 0]
            concentrations = preprocessing.inverse_target(station_predictions.cpu().numpy(), clip=False)

            if not np.isfinite(concentrations).all():
                raise ValueError("Nonfinite evaluation predictions")

            columns = ["station_uuid", "time_utc", "node_index", "value_ugm3"]
            frame = batch["labels"][columns].rename(columns={"value_ugm3": "observed"}).copy()
            frame["predicted"] = concentrations.clip(min=0)
            frame["clipped"] = concentrations < 0
            split_frames.append(frame)

    prediction_tables[split] = pd.concat(split_frames, ignore_index=True)

## Metric calculation


In [ ]:
def calculate_metrics(frame):
    observed = frame["observed"].to_numpy()
    predicted = frame["predicted"].to_numpy()
    error = predicted - observed

    mae = np.abs(error).mean()
    rmse = np.sqrt(np.mean(error ** 2))
    bias = error.mean()
    mean_observed = observed.mean()
    total_variation = np.sum((observed - mean_observed) ** 2)
    r2 = 1 - np.sum(error ** 2) / total_variation if total_variation > 0 else np.nan
    nrmse = rmse / mean_observed if mean_observed != 0 else np.nan

    return {
        "n": len(frame),
        "mae": float(mae),
        "rmse": float(rmse),
        "bias": float(bias),
        "r2": float(r2),
        "mean_obs": float(mean_observed),
        "nrmse": float(nrmse),
        "clipped_fraction": float(frame["clipped"].mean()),
    }

## Overall metrics


In [ ]:
overall_rows = []
for split, predictions in prediction_tables.items():
    row = {"split": split, "station_uuid": "pooled"}
    row.update(calculate_metrics(predictions))
    overall_rows.append(row)

overall_metrics = pd.DataFrame(overall_rows)
display(overall_metrics)

## Station metrics


In [ ]:
station_rows = []
for split, predictions in prediction_tables.items():
    for station_uuid, station_predictions in predictions.groupby("station_uuid", sort=True):
        row = {"split": split, "station_uuid": station_uuid}
        row.update(calculate_metrics(station_predictions))
        station_rows.append(row)

station_metrics = pd.DataFrame(station_rows)
display(station_metrics.head())

## Save evaluation


In [ ]:
for split, predictions in prediction_tables.items():
    predictions.to_parquet(OUTPUT_DIR / f"{split}_predictions.parquet", index=False)

metrics = pd.concat([overall_metrics, station_metrics], ignore_index=True)
metrics.to_csv(OUTPUT_DIR / "evaluation_metrics.csv", index=False)
print("Saved:", OUTPUT_DIR.resolve())